# Principal Component Analysis (PCA) application to Chroma Keying

## Previous Work
In our previous work, we explored various techniques for chroma keying, including color space transformations and thresholding methods. We found that while these techniques can be effective, they often struggle with complex backgrounds and varying lighting conditions.

### Approach 1

In [2]:
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from ipywidgets import IntSlider, FloatSlider, Button, Output, VBox, HBox, FileUpload, HTML
from IPython.display import display, clear_output
import io

# Create interactive chroma keying simulator - Approach 1
print("=== Interactive Chroma Keying Simulator - Approach 1 ===")
print("Adjust RGB thresholds to remove a color from the image\n")

# Create sliders for RGB min/max thresholds
min_r = IntSlider(value=0, min=0, max=255, step=1, description='Min R:')
min_g = IntSlider(value=75, min=0, max=255, step=1, description='Min G:')
min_b = IntSlider(value=0, min=0, max=255, step=1, description='Min B:')

max_r = IntSlider(value=100, min=0, max=255, step=1, description='Max R:')
max_g = IntSlider(value=255, min=0, max=255, step=1, description='Max G:')
max_b = IntSlider(value=100, min=0, max=255, step=1, description='Max B:')

# File upload widget
file_upload = FileUpload(accept='image/*', multiple=False, description='Upload Image')

# Process button
process_btn = Button(description='Process Image', button_style='info')

# Output display
output_display = Output()

# Store current image
current_image = {'data': None}

def process_chroma_key():
    """Process image with chroma keying based on Approach 1"""
    if current_image['data'] is None:
        with output_display:
            clear_output()
            print("Please upload an image first")
        return
    
    img = current_image['data']
    img_array = np.array(img)
    
    # Convert to float (0-1) if needed
    if img_array.max() > 1:
        img_array = img_array.astype(np.float32) / 255.0
    
    # Get thresholds
    min_vals = np.array([min_r.value, min_g.value, min_b.value]) / 255.0
    max_vals = np.array([max_r.value, max_g.value, max_b.value]) / 255.0
    
    # Handle RGB vs RGBA
    if img_array.shape[2] == 4:  # RGBA
        rgb = img_array[:, :, :3]
        alpha = img_array[:, :, 3:]
    else:  # RGB
        rgb = img_array
        alpha = None
    
    # Create mask: True where pixel is within threshold (chroma key color)
    mask = np.all((rgb >= min_vals) & (rgb <= max_vals), axis=2)
    
    # Create result: white where mask is True, original otherwise
    result = rgb.copy()
    result[mask] = [1, 1, 1]  # White background
    
    # Display results
    with output_display:
        clear_output()
        fig, axes = plt.subplots(1, 2, figsize=(12, 5))
        
        # Original image
        axes[0].imshow(rgb)
        axes[0].set_title('Original Image')
        axes[0].axis('off')
        
        # Processed image
        axes[1].imshow(result)
        axes[1].set_title('Chroma Keyed (Approach 1)')
        axes[1].axis('off')
        
        plt.tight_layout()
        plt.show()
        
        # Show statistics
        keyed_pixels = np.sum(mask)
        total_pixels = mask.shape[0] * mask.shape[1]
        percentage = (keyed_pixels / total_pixels) * 100
        print(f"\nChroma Key Statistics:")
        print(f"Total pixels: {total_pixels}")
        print(f"Keyed pixels (removed): {keyed_pixels}")
        print(f"Percentage removed: {percentage:.2f}%")

def handle_file_upload(change):
    """Handle image file upload"""
    if file_upload.value:
        uploaded_value = file_upload.value
        uploaded_file = list(uploaded_value.values())[0] if hasattr(uploaded_value, 'values') else uploaded_value[0]
        img = Image.open(io.BytesIO(uploaded_file['content'])).convert('RGBA')
        current_image['data'] = img
        
        with output_display:
            clear_output()
            print(f"✓ Image loaded: {uploaded_file['name']} ({img.size[0]}x{img.size[1]})")

def handle_process_click(button):
    """Handle process button click"""
    process_chroma_key()

file_upload.observe(handle_file_upload, names='value')
process_btn.on_click(handle_process_click)

# Create layout
controls = VBox([
    HTML("<b>Color Thresholds (RGB)</b>"),
    HTML("<b>Minimum Values:</b>"),
    min_r, min_g, min_b,
    HTML("<b>Maximum Values:</b>"),
    max_r, max_g, max_b,
    file_upload, process_btn
])

display(HBox([controls, output_display]))

print("\nInstructions:")
print("1. Upload an image (typically one with a green or blue background)")
print("2. Adjust the RGB thresholds to match the background color")
print("3. Click 'Process Image' to apply chroma keying")
print("4. The keyed color will be replaced with white")

=== Interactive Chroma Keying Simulator - Approach 1 ===
Adjust RGB thresholds to remove a color from the image




Instructions:
1. Upload an image (typically one with a green or blue background)
2. Adjust the RGB thresholds to match the background color
3. Click 'Process Image' to apply chroma keying
4. The keyed color will be replaced with white


### Approach 2

In [ ]:
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from ipywidgets import Button, FileUpload, Output, VBox, HBox, HTML
from IPython.display import display, clear_output
import io

print("=== Interactive Chroma Keying Simulator - Approach 2 ===")
print("Remove pixels where green is the dominant color channel.\n")

file_upload_2 = FileUpload(accept='image/*', multiple=False, description='Upload Image')
process_btn_2 = Button(description='Process Image', button_style='info')
output_display_2 = Output()
current_image_2 = {'data': None}

def process_chroma_key_2():
    if current_image_2['data'] is None:
        with output_display_2:
            clear_output()
            print("Please upload an image first")
        return

    img = current_image_2['data']
    img_array = np.array(img)

    if img_array.max() > 1:
        img_array = img_array.astype(np.float32) / 255.0

    if img_array.shape[2] == 4:
        rgb = img_array[:, :, :3]
    else:
        rgb = img_array

    mask = np.argmax(rgb, axis=2) == 1

    result = rgb.copy()
    result[mask] = [1, 1, 1]

    with output_display_2:
        clear_output()
        fig, axes = plt.subplots(1, 2, figsize=(12, 5))
        axes[0].imshow(rgb)
        axes[0].set_title('Original Image')
        axes[0].axis('off')

        axes[1].imshow(result)
        axes[1].set_title('Chroma Keyed (Approach 2)')
        axes[1].axis('off')

        plt.tight_layout()
        plt.show()

        keyed_pixels = np.sum(mask)
        total_pixels = mask.shape[0] * mask.shape[1]
        percentage = (keyed_pixels / total_pixels) * 100
        print("\nChroma Key Statistics:")
        print(f"Total pixels: {total_pixels}")
        print(f"Keyed pixels (removed): {keyed_pixels}")
        print(f"Percentage removed: {percentage:.2f}%")

def handle_file_upload_2(change):
    if file_upload_2.value:
        uploaded_value = file_upload_2.value
        uploaded_file = list(uploaded_value.values())[0] if hasattr(uploaded_value, 'values') else uploaded_value[0]
        img = Image.open(io.BytesIO(uploaded_file['content'])).convert('RGBA')
        current_image_2['data'] = img

        with output_display_2:
            clear_output()
            print(f"Image loaded: {uploaded_file['name']} ({img.size[0]}x{img.size[1]})")

def handle_process_click_2(button):
    process_chroma_key_2()

file_upload_2.observe(handle_file_upload_2, names='value')
process_btn_2.on_click(handle_process_click_2)

controls_2 = VBox([
    HTML("<b>Approach 2</b>"),
    HTML("Remove pixels where green is the largest RGB component."),
    file_upload_2,
    process_btn_2
])

display(HBox([controls_2, output_display_2]))

print("\nInstructions:")
print("1. Upload an image with a green screen or green-dominant background")
print("2. Click 'Process Image' to remove pixels where green is dominant")
print("3. The keyed color will be replaced with white")

### Approach 3

In [ ]:
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from ipywidgets import FloatSlider, Button, FileUpload, Output, VBox, HBox, HTML
from IPython.display import display, clear_output
import io

print("=== Interactive Chroma Keying Simulator - Approach 3 ===")
print("Use green dominance with lower/upper bounds and a color-difference threshold.\n")

green_lower = FloatSlider(value=0.28, min=0, max=1, step=0.01, description='Green Min:')
green_upper = FloatSlider(value=0.97, min=0, max=1, step=0.01, description='Green Max:')
grey_bound = FloatSlider(value=0.00, min=0, max=1, step=0.005, description='Diff Min:')

file_upload_3 = FileUpload(accept='image/*', multiple=False, description='Upload Image')
process_btn_3 = Button(description='Process Image', button_style='info')
output_display_3 = Output()
current_image_3 = {'data': None}

def process_chroma_key_3():
    if current_image_3['data'] is None:
        with output_display_3:
            clear_output()
            print("Please upload an image first")
        return

    img = current_image_3['data']
    img_array = np.array(img)

    if img_array.max() > 1:
        img_array = img_array.astype(np.float32) / 255.0

    if img_array.shape[2] == 4:
        rgb = img_array[:, :, :3]
    else:
        rgb = img_array

    r = rgb[:, :, 0]
    g = rgb[:, :, 1]
    b = rgb[:, :, 2]

    mask = (
        (np.argmax(rgb, axis=2) == 1)
        & (g > green_lower.value)
        & (g <= green_upper.value)
        & (np.abs(r - b) >= grey_bound.value)
        & (np.abs(r - g) >= grey_bound.value)
        & (np.abs(b - g) >= grey_bound.value)
    )

    result = rgb.copy()
    result[mask] = [1, 1, 1]

    with output_display_3:
        clear_output()
        fig, axes = plt.subplots(1, 2, figsize=(12, 5))
        axes[0].imshow(rgb)
        axes[0].set_title('Original Image')
        axes[0].axis('off')

        axes[1].imshow(result)
        axes[1].set_title('Chroma Keyed (Approach 3)')
        axes[1].axis('off')

        plt.tight_layout()
        plt.show()

        keyed_pixels = np.sum(mask)
        total_pixels = mask.shape[0] * mask.shape[1]
        percentage = (keyed_pixels / total_pixels) * 100
        print("\nChroma Key Statistics:")
        print(f"Total pixels: {total_pixels}")
        print(f"Keyed pixels (removed): {keyed_pixels}")
        print(f"Percentage removed: {percentage:.2f}%")

def handle_file_upload_3(change):
    if file_upload_3.value:
        uploaded_value = file_upload_3.value
        uploaded_file = list(uploaded_value.values())[0] if hasattr(uploaded_value, 'values') else uploaded_value[0]
        img = Image.open(io.BytesIO(uploaded_file['content'])).convert('RGBA')
        current_image_3['data'] = img

        with output_display_3:
            clear_output()
            print(f"Image loaded: {uploaded_file['name']} ({img.size[0]}x{img.size[1]})")

def handle_process_click_3(button):
    process_chroma_key_3()

file_upload_3.observe(handle_file_upload_3, names='value')
process_btn_3.on_click(handle_process_click_3)

controls_3 = VBox([
    HTML("<b>Approach 3</b>"),
    HTML("Tune green bounds and a minimum color separation threshold."),
    green_lower,
    green_upper,
    grey_bound,
    file_upload_3,
    process_btn_3
])

display(HBox([controls_3, output_display_3]))

print("\nInstructions:")
print("1. Upload an image with a green background")
print("2. Adjust the green bounds and minimum color difference threshold")
print("3. Click 'Process Image' to apply chroma keying")
print("4. The keyed color will be replaced with white")

=== Interactive Chroma Keying Simulator - Approach 3 ===
Use green dominance with lower/upper bounds and a color-difference threshold.




Instructions:
1. Upload an image with a green background
2. Adjust the green bounds and minimum color difference threshold
3. Click 'Process Image' to apply chroma keying
4. The keyed color will be replaced with white


## New Approach: PCA for Chroma Keying

Principal Component Analysis (PCA) is a powerful statistical technique that can be used to reduce the dimensionality of data while preserving as much variance as possible. In the context of chroma keying, PCA can help us identify the most significant color components in an image, allowing us to effectively separate the foreground from the background.

### Distribution of RGB

To apply PCA to our chroma keying problem, we first need to analyze the distribution of RGB values in our images. By plotting the RGB values in a 3D space, we can visualize how the colors are distributed and identify clusters that correspond to the foreground and background.

Consider the following example image:

In [ ]:
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from ipywidgets import Button, FileUpload, Output, VBox, HBox, HTML, Tab
from IPython.display import display, clear_output
import io
from matplotlib.colors import to_rgb

print("=== Image Display & RGB Analysis Visualization ===\n")

# File upload widget
file_upload_viz = FileUpload(accept='image/*', multiple=False, description='Upload Image')
process_btn_viz = Button(description='Analyze Image', button_style='success')
output_display_viz = Output()
current_image_viz = {'data': None, 'name': None}

def process_image_visualization():
    """Process and visualize image with RGB histogram and other visualizations"""
    if current_image_viz['data'] is None:
        with output_display_viz:
            clear_output()
            print("Please upload an image first")
        return
    
    img = current_image_viz['data']
    img_name = current_image_viz['name']
    img_array = np.array(img)
    
    # Convert to float (0-1) if needed
    if img_array.max() > 1:
        img_array = img_array.astype(np.float32) / 255.0
    
    # Handle RGB vs RGBA
    if img_array.shape[2] == 4:
        rgb = img_array[:, :, :3]
    else:
        rgb = img_array
    
    # Reshape to get all pixels
    h, w, c = rgb.shape
    pixels = rgb.reshape(-1, 3)
    
    with output_display_viz:
        clear_output()
        
        # Create figure with subplots
        fig = plt.figure(figsize=(16, 12))
        
        # 1. Original Image
        ax1 = plt.subplot(3, 3, 1)
        ax1.imshow(rgb)
        ax1.set_title(f'Original Image\n({w}x{h})', fontsize=12, fontweight='bold')
        ax1.axis('off')
        
        # 2. Red Channel
        ax2 = plt.subplot(3, 3, 2)
        ax2.imshow(rgb[:, :, 0], cmap='Reds')
        ax2.set_title('Red Channel', fontsize=12, fontweight='bold')
        ax2.axis('off')
        
        # 3. Green Channel
        ax3 = plt.subplot(3, 3, 3)
        ax3.imshow(rgb[:, :, 1], cmap='Greens')
        ax3.set_title('Green Channel', fontsize=12, fontweight='bold')
        ax3.axis('off')
        
        # 4. Blue Channel
        ax4 = plt.subplot(3, 3, 4)
        ax4.imshow(rgb[:, :, 2], cmap='Blues')
        ax4.set_title('Blue Channel', fontsize=12, fontweight='bold')
        ax4.axis('off')
        
        # 5. RGB Histogram
        ax5 = plt.subplot(3, 3, 5)
        bins = 256 if rgb.max() > 1 else 256
        ax5.hist(pixels[:, 0].flatten(), bins=bins, color='red', alpha=0.5, label='Red', density=True)
        ax5.hist(pixels[:, 1].flatten(), bins=bins, color='green', alpha=0.5, label='Green', density=True)
        ax5.hist(pixels[:, 2].flatten(), bins=bins, color='blue', alpha=0.5, label='Blue', density=True)
        ax5.set_xlabel('Pixel Value')
        ax5.set_ylabel('Density')
        ax5.set_title('RGB Histogram (Overlaid)', fontsize=12, fontweight='bold')
        ax5.legend()
        ax5.grid(alpha=0.3)
        
        # 6. Red Histogram
        ax6 = plt.subplot(3, 3, 6)
        ax6.bar(range(len(np.histogram(pixels[:, 0], bins=256)[0])), 
               np.histogram(pixels[:, 0], bins=256)[0], 
               color='red', alpha=0.7, width=1.0)
        ax6.set_xlabel('Red Value')
        ax6.set_ylabel('Frequency')
        ax6.set_title('Red Channel Histogram', fontsize=12, fontweight='bold')
        ax6.grid(alpha=0.3)
        
        # 7. Green Histogram
        ax7 = plt.subplot(3, 3, 7)
        ax7.bar(range(len(np.histogram(pixels[:, 1], bins=256)[0])), 
               np.histogram(pixels[:, 1], bins=256)[0], 
               color='green', alpha=0.7, width=1.0)
        ax7.set_xlabel('Green Value')
        ax7.set_ylabel('Frequency')
        ax7.set_title('Green Channel Histogram', fontsize=12, fontweight='bold')
        ax7.grid(alpha=0.3)
        
        # 8. Blue Histogram
        ax8 = plt.subplot(3, 3, 8)
        ax8.bar(range(len(np.histogram(pixels[:, 2], bins=256)[0])), 
               np.histogram(pixels[:, 2], bins=256)[0], 
               color='blue', alpha=0.7, width=1.0)
        ax8.set_xlabel('Blue Value')
        ax8.set_ylabel('Frequency')
        ax8.set_title('Blue Channel Histogram', fontsize=12, fontweight='bold')
        ax8.grid(alpha=0.3)
        
        # 9. RGB 3D Scatter Plot (sampled for performance)
        ax9 = fig.add_subplot(3, 3, 9, projection='3d')
        sample_indices = np.random.choice(pixels.shape[0], size=min(5000, pixels.shape[0]), replace=False)
        sampled_pixels = pixels[sample_indices]
        
        scatter = ax9.scatter(sampled_pixels[:, 0], sampled_pixels[:, 1], sampled_pixels[:, 2], 
                             c=sampled_pixels, s=1, alpha=0.6)
        ax9.set_xlabel('Red')
        ax9.set_ylabel('Green')
        ax9.set_zlabel('Blue')
        ax9.set_title('RGB Color Space\n(Sampled)', fontsize=12, fontweight='bold')
        
        plt.tight_layout()
        plt.show()
        
        # Print statistics
        print(f"\n📊 Image Analysis Statistics for: {img_name}")
        print("=" * 60)
        print(f"Image Dimensions: {w}x{h} pixels ({h*w:,} total pixels)")
        print(f"Image Size: {img_array.nbytes / (1024*1024):.2f} MB")
        print(f"\nChannel Statistics:")
        print(f"  Red   - Mean: {pixels[:, 0].mean()*255:.2f}, Std: {pixels[:, 0].std()*255:.2f}, "
              f"Range: [{pixels[:, 0].min()*255:.2f}, {pixels[:, 0].max()*255:.2f}]")
        print(f"  Green - Mean: {pixels[:, 1].mean()*255:.2f}, Std: {pixels[:, 1].std()*255:.2f}, "
              f"Range: [{pixels[:, 1].min()*255:.2f}, {pixels[:, 1].max()*255:.2f}]")
        print(f"  Blue  - Mean: {pixels[:, 2].mean()*255:.2f}, Std: {pixels[:, 2].std()*255:.2f}, "
              f"Range: [{pixels[:, 2].min()*255:.2f}, {pixels[:, 2].max()*255:.2f}]")
        
        # Calculate dominant color
        mean_color = pixels.mean(axis=0) * 255
        print(f"\nMean Color (RGB): ({mean_color[0]:.0f}, {mean_color[1]:.0f}, {mean_color[2]:.0f})")
        
        # Color distribution analysis
        r_hist, _ = np.histogram(pixels[:, 0], bins=256)
        g_hist, _ = np.histogram(pixels[:, 1], bins=256)
        b_hist, _ = np.histogram(pixels[:, 2], bins=256)
        
        print(f"\nColor Distribution:")
        print(f"  Red   - Dominant Range: {np.argmax(r_hist)}-{np.argmax(r_hist)+1}")
        print(f"  Green - Dominant Range: {np.argmax(g_hist)}-{np.argmax(g_hist)+1}")
        print(f"  Blue  - Dominant Range: {np.argmax(b_hist)}-{np.argmax(b_hist)+1}")

def handle_file_upload_viz(change):
    """Handle image file upload"""
    if file_upload_viz.value:
        uploaded_value = file_upload_viz.value
        uploaded_file = list(uploaded_value.values())[0] if hasattr(uploaded_value, 'values') else uploaded_value[0]
        img = Image.open(io.BytesIO(uploaded_file['content'])).convert('RGB')
        current_image_viz['data'] = img
        current_image_viz['name'] = uploaded_file['name']
        
        with output_display_viz:
            clear_output()
            print(f"✓ Image loaded: {uploaded_file['name']}\nSize: {img.size[0]}x{img.size[1]} pixels")
            print("Click 'Analyze Image' to view visualizations")

def handle_process_click_viz(button):
    """Handle process button click"""
    process_image_visualization()

file_upload_viz.observe(handle_file_upload_viz, names='value')
process_btn_viz.on_click(handle_process_click_viz)

# Create layout
controls_viz = VBox([
    HTML("<b style='font-size:14px'>📈 Image Analysis & Visualization</b>"),
    file_upload_viz,
    process_btn_viz,
    HTML("<br><i>Upload an image to see:</i>"),
    HTML("<ul><li>Original image and individual color channels</li><li>RGB histograms</li><li>3D color space visualization</li><li>Channel statistics</li></ul>")
])

display(HBox([controls_viz, output_display_viz]))

print("\n✅ Image visualization tool ready!")

=== Image Display & RGB Analysis Visualization ===




✅ Image visualization tool ready!


## PCA-Based Chroma Keying

PCA can help with green screen removal by modeling how background colors vary instead of relying on fixed RGB thresholds.

The main idea is:

1. We assume the border of the image mostly contains the green screen background.
2. We collect those border pixels and treat each pixel as a 3D point `(R, G, B)`.
3. We compute the mean background color and apply PCA to the border-pixel cloud.
4. PCA gives us the main directions in which the background color changes because of lighting, shadows, wrinkles, and camera noise.
5. For every pixel in the image, we measure how far it is from the learned background distribution in PCA space.
6. Pixels that are close to the background model are marked as background and removed.

Why this is useful:

- Simple thresholding uses fixed cutoffs like `G > 0.7`, which can fail when the green screen is unevenly lit.
- PCA learns the shape of the background color cluster.
- That makes it more adaptive when the green background has gradients, darker corners, or spill.

In this notebook, we use PCA as a background model, not only as a visualization tool.

In [ ]:
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from ipywidgets import FloatSlider, Dropdown, Button, FileUpload, Output, VBox, HBox, HTML
from IPython.display import display, clear_output
import io

print("=== PCA-Based Chroma Keying ===")
print("Model the background color distribution with PCA and remove pixels close to that model.\n")

file_upload_pca = FileUpload(accept='image/*', multiple=False, description='Upload Image')
process_btn_pca = Button(description='Run PCA Keying', button_style='success')
output_display_pca = Output()

background_source = Dropdown(
    options=[('Border pixels from main image', 'border'), ('Approach 3 complement from main image', 'approach3')],
    value='approach3',
    description='Fit from:'
)

border_fraction = FloatSlider(value=0.10, min=0.02, max=0.25, step=0.01, description='Border:')
distance_threshold = FloatSlider(value=2.50, min=0.50, max=6.00, step=0.10, description='Threshold:')
green_lower_pca = FloatSlider(value=0.28, min=0, max=1, step=0.01, description='Green Min:')
green_upper_pca = FloatSlider(value=0.97, min=0, max=1, step=0.01, description='Green Max:')
grey_bound_pca = FloatSlider(value=0.00, min=0, max=1, step=0.005, description='Diff Min:')
replacement_mode = Dropdown(
    options=[('White background', 'white'), ('Transparent background', 'transparent')],
    value='white',
    description='Output:'
)

current_image_pca = {'data': None, 'name': None}

def get_uploaded_file(upload_value):
    return list(upload_value.values())[0] if hasattr(upload_value, 'values') else upload_value[0]

def extract_rgb_array(img):
    img_array = np.array(img).astype(np.float32) / 255.0
    return img_array[:, :, :3]

def get_border_pixels(rgb, frac):
    h, w, _ = rgb.shape
    bh = max(1, int(h * frac))
    bw = max(1, int(w * frac))

    top = rgb[:bh, :, :].reshape(-1, 3)
    bottom = rgb[-bh:, :, :].reshape(-1, 3)
    left = rgb[bh:h-bh, :bw, :].reshape(-1, 3) if h > 2 * bh else np.empty((0, 3))
    right = rgb[bh:h-bh, -bw:, :].reshape(-1, 3) if h > 2 * bh else np.empty((0, 3))

    return np.vstack([top, bottom, left, right])

def get_approach3_background_mask(rgb, green_lower, green_upper, grey_bound):
    r = rgb[:, :, 0]
    g = rgb[:, :, 1]
    b = rgb[:, :, 2]
    return (
        (np.argmax(rgb, axis=2) == 1)
        & (g > green_lower)
        & (g <= green_upper)
        & (np.abs(r - b) >= grey_bound)
        & (np.abs(r - g) >= grey_bound)
        & (np.abs(b - g) >= grey_bound)
    )

def fit_background_pca(border_pixels):
    mean_bg = border_pixels.mean(axis=0)
    centered = border_pixels - mean_bg
    cov = np.cov(centered, rowvar=False)
    eigvals, eigvecs = np.linalg.eigh(cov)
    order = np.argsort(eigvals)[::-1]
    eigvals = eigvals[order]
    eigvecs = eigvecs[:, order]
    stds = np.sqrt(np.maximum(eigvals, 1e-8))
    return mean_bg, eigvecs, eigvals, stds

def pca_background_distance(pixels, mean_bg, eigvecs, stds):
    centered = pixels - mean_bg
    projected = centered @ eigvecs
    normalized = projected / stds
    return np.sqrt(np.sum(normalized ** 2, axis=1)), projected

def apply_pca_chroma_key(rgb, frac, threshold, output_mode='white', fit_pixels=None, source_label='border pixels', seed_mask=None):
    background_pixels = get_border_pixels(rgb, frac) if fit_pixels is None else fit_pixels
    mean_bg, eigvecs, eigvals, stds = fit_background_pca(background_pixels)

    h, w, _ = rgb.shape
    pixels = rgb.reshape(-1, 3)
    distances, projected = pca_background_distance(pixels, mean_bg, eigvecs, stds)
    mask = distances.reshape(h, w) <= threshold

    if output_mode == 'transparent':
        alpha = np.ones((h, w, 1), dtype=np.float32)
        alpha[mask] = 0.0
        result = np.dstack([rgb, alpha])
    else:
        result = rgb.copy()
        result[mask] = [1.0, 1.0, 1.0]

    return {
        'result': result,
        'mask': mask,
        'distances': distances.reshape(h, w),
        'mean_bg': mean_bg,
        'eigvecs': eigvecs,
        'eigvals': eigvals,
        'background_pixels': background_pixels,
        'projected_pixels': projected,
        'source_label': source_label,
        'seed_mask': seed_mask,
    }

def process_pca_keying():
    if current_image_pca['data'] is None:
        with output_display_pca:
            clear_output()
            print('Please upload an image first')
        return

    rgb = extract_rgb_array(current_image_pca['data'])
    fit_pixels = None
    source_label = 'border pixels from main image'
    seed_mask = None
    if background_source.value == 'approach3':
        seed_mask = get_approach3_background_mask(
            rgb,
            green_lower=green_lower_pca.value,
            green_upper=green_upper_pca.value,
            grey_bound=grey_bound_pca.value,
        )
        fit_pixels = rgb[seed_mask]
        if fit_pixels.shape[0] < 25:
            with output_display_pca:
                clear_output()
                print('Approach 3 found too few background pixels to fit PCA reliably. Relax the Approach 3 thresholds or switch Fit from to border pixels.')
            return
        source_label = 'Approach 3 estimated background pixels from main image'
    analysis = apply_pca_chroma_key(
        rgb,
        frac=border_fraction.value,
        threshold=distance_threshold.value,
        output_mode=replacement_mode.value,
        fit_pixels=fit_pixels,
        source_label=source_label,
        seed_mask=seed_mask,
    )

    mask = analysis['mask']
    result = analysis['result']
    distances = analysis['distances']
    background_pixels = analysis['background_pixels']
    projected_pixels = analysis['projected_pixels']
    eigvals = analysis['eigvals']
    mean_bg = analysis['mean_bg']
    source_label = analysis['source_label']
    seed_mask = analysis['seed_mask']

    with output_display_pca:
        clear_output()
        fig = plt.figure(figsize=(16, 10))

        ax1 = plt.subplot(2, 3, 1)
        ax1.imshow(rgb)
        ax1.set_title('Original Image')
        ax1.axis('off')

        ax2 = plt.subplot(2, 3, 2)
        if seed_mask is None:
            ax2.imshow(mask, cmap='gray')
            ax2.set_title('Background Mask (PCA)')
        else:
            ax2.imshow(seed_mask, cmap='gray')
            ax2.set_title('Approach 3 Seed Mask')
        ax2.axis('off')

        ax3 = plt.subplot(2, 3, 3)
        ax3.imshow(result)
        ax3.set_title('Chroma Key Result')
        ax3.axis('off')

        ax4 = plt.subplot(2, 3, 4)
        im = ax4.imshow(distances, cmap='viridis')
        ax4.set_title('Distance from Background Model')
        ax4.axis('off')
        plt.colorbar(im, ax=ax4, fraction=0.046, pad=0.04)

        ax5 = plt.subplot(2, 3, 5)
        sample_size = min(4000, background_pixels.shape[0])
        border_idx = np.random.choice(background_pixels.shape[0], size=sample_size, replace=False)
        sampled_border = background_pixels[border_idx]
        ax5.scatter(sampled_border[:, 0], sampled_border[:, 1], c=sampled_border, s=4, alpha=0.6)
        ax5.set_xlabel('Red')
        ax5.set_ylabel('Green')
        ax5.set_title('Background Training Pixels (R vs G)')
        ax5.grid(alpha=0.3)

        ax6 = plt.subplot(2, 3, 6)
        sample_size_all = min(5000, projected_pixels.shape[0])
        proj_idx = np.random.choice(projected_pixels.shape[0], size=sample_size_all, replace=False)
        proj_sample = projected_pixels[proj_idx]
        rgb_sample = rgb.reshape(-1, 3)[proj_idx]
        ax6.scatter(proj_sample[:, 0], proj_sample[:, 1], c=rgb_sample, s=4, alpha=0.5)
        ax6.set_xlabel('PC1')
        ax6.set_ylabel('PC2')
        ax6.set_title('Pixels in PCA Space')
        ax6.grid(alpha=0.3)

        plt.tight_layout()
        plt.show()

        total_pixels = mask.size
        keyed_pixels = int(mask.sum())
        explained = eigvals / np.maximum(eigvals.sum(), 1e-8)

        print('\nPCA Background Model Summary')
        print('=' * 40)
        print(f"Image: {current_image_pca['name']}")
        print(f"PCA fit source: {source_label}")
        print(f"Estimated background mean RGB: ({mean_bg[0]*255:.1f}, {mean_bg[1]*255:.1f}, {mean_bg[2]*255:.1f})")
        print(f"Explained variance ratio: PC1={explained[0]:.3f}, PC2={explained[1]:.3f}, PC3={explained[2]:.3f}")
        print(f"Background pixels removed: {keyed_pixels:,} / {total_pixels:,} ({100 * keyed_pixels / total_pixels:.2f}%)")
        if background_source.value == 'border':
            print(f"Border fraction used for PCA fit: {border_fraction.value:.2f}")
        if background_source.value == 'approach3':
            print(f"Approach 3 seed pixels used for PCA fit: {background_pixels.shape[0]:,}")
            print(f"Approach 3 params: green in ({green_lower_pca.value:.2f}, {green_upper_pca.value:.2f}], diff >= {grey_bound_pca.value:.3f}")
        print(f"Distance threshold: {distance_threshold.value:.2f}")

        print('\nHow to tune it:')
        print('- Increase Border if the image edges are mostly green screen and you want a stronger background model.')
        print('- Decrease Border if the subject touches the image edges.')
        print('- Increase Threshold to remove more background.')
        print('- Decrease Threshold to protect the foreground.')

def handle_file_upload_pca(change):
    if file_upload_pca.value:
        uploaded_file = get_uploaded_file(file_upload_pca.value)
        img = Image.open(io.BytesIO(uploaded_file['content'])).convert('RGBA')
        current_image_pca['data'] = img
        current_image_pca['name'] = uploaded_file['name']

        with output_display_pca:
            clear_output()
            print(f"Image loaded: {uploaded_file['name']} ({img.size[0]}x{img.size[1]})")
            print('Click "Run PCA Keying" to analyze and remove the background.')

def handle_process_click_pca(button):
    process_pca_keying()

file_upload_pca.observe(handle_file_upload_pca, names='value')
process_btn_pca.on_click(handle_process_click_pca)

controls_pca = VBox([
    HTML('<b>PCA-Based Chroma Keying</b>'),
    HTML('Fit a PCA model on either border pixels or on Approach 3 estimated background pixels from the same image, then remove colors close to that background distribution.'),
    background_source,
    border_fraction,
    distance_threshold,
    green_lower_pca,
    green_upper_pca,
    grey_bound_pca,
    replacement_mode,
    file_upload_pca,
    process_btn_pca,
])

display(HBox([controls_pca, output_display_pca]))

print('Instructions:')
print('1. Upload the main green screen image.')
print('2. Choose whether PCA should fit from border pixels or from Approach 3 estimated background pixels.')
print('3. If using border pixels, tune the border fraction. If using Approach 3, tune the green bounds and difference threshold used to generate the seed background mask.')
print('4. Use the distance threshold to control how aggressively background-like pixels are removed.')
print('5. Run PCA keying and compare the seed mask, PCA result, and final output.')

=== PCA-Based Chroma Keying ===
Model the background color distribution with PCA and remove pixels close to that model.



Instructions:
1. Upload the main green screen image.
2. Choose whether PCA should fit from border pixels or from Approach 3 estimated background pixels.
3. If using border pixels, tune the border fraction. If using Approach 3, tune the green bounds and difference threshold used to generate the seed background mask.
4. Use the distance threshold to control how aggressively background-like pixels are removed.
5. Run PCA keying and compare the seed mask, PCA result, and final output.


## AKMC Project Report Explained

This project studies chroma keying as a pixel classification problem: for each pixel with color `(r, g, b)`, decide whether it belongs to the green-screen background or to the foreground object.

### Overall Methodology

The report follows this progression:

1. Observe the RGB distributions of sample images using histograms and basic summaries.
2. Propose simple rules that describe how green-screen pixels differ from foreground pixels.
3. Test those rules on real images.
4. Refine the rules to make them less sensitive to lighting and scale.
5. Identify which quantities are more intrinsic to the green screen itself.

Mathematically, if `x = (r, g, b)` is a pixel, the task is to define a decision rule

`f(x) = 1` if the pixel is background, and `f(x) = 0` otherwise.

### Approach Demi: Histogram Screening

Idea: estimate background ranges from the RGB histograms, then remove pixels inside those ranges.

Rule:

`f(r,g,b) = 1` if `r in [r_min, r_max]`, `g in [g_min, g_max]`, and `b in [b_min, b_max]`.

Logic:

- Background occupies a large, fairly uniform region of color space.
- So histogram peaks can be used to guess the green-screen color interval.

Drawback:

- The cutoff is unstable and hard to automate across frames.

### Approach Grande: Smarter RGB Bounds

Idea: use the fact that green-screen pixels should have high green and relatively low red/blue.

Rule:

`f(r,g,b) = 1` if `g > t_g`, `r < t_r`, and `b < t_b`.

Example from the report: `g > 0.35`, `r < 0.5`, `b < 0.5`.

Logic:

- This is better than pure histogram slicing because it encodes a structural property of green.

Drawback:

- Thresholds were found mainly by hit-and-trial.

### Approach Venti: Ratio-Based Screening

Idea: instead of absolute intensities, compare green relative to red and blue.

Rule:

`f(r,g,b) = 1` if `g/r > s` and `g/b > s`, where `s` is the super-bound.

Typical value in the report: `s = 1.2`.

Logic:

- Ratios are more stable than raw values under moderate illumination changes.
- If the whole background gets brighter or darker, the ratio may remain similar even when the raw RGB values change.

Drawback:

- A single ratio threshold may still remove some foreground or leave some background.

### Approach Trenta: Intrinsic Green-Screen Bound

Idea: measure the minimum green dominance on a blank green screen, then use a safety factor.

If the blank screen has

`s_r = min(g/r)` and `s_b = min(g/b)`,

then define a conservative bound

`b_bound = alpha * min(s_r, s_b)` with `alpha = 0.9` in the report.

Rule:

`f(r,g,b) = 1` if `g/r > b_bound` and `g/b > b_bound`.

Logic:

- The report argues that this bound is closer to an intrinsic property of the physical green screen than absolute RGB thresholds.
- Ratios capture green dominance while being less affected by lighting scale.

### Additional Logical Refinement: What Is Green?

The report notices that ratio conditions alone are not enough. A pixel can satisfy large `g/r` and `g/b` ratios and still not be perceived as green if all channels are very dark.

So a better practical rule is:

`f(r,g,b) = 1` if `g/r > s`, `g/b > s`, and the pixel is actually green enough to be visually considered green.

In code, we approximate that with conditions like:

- `g > g_min`
- `g > r` and `g > b`

### Conclusion of the Report

The report concludes that ratio-based chroma keying is more robust than absolute RGB thresholding. In particular:

- Histogram-only bounds are too unstable.
- Direct RGB bounds are better, but still image-dependent.
- Ratio bounds `g/r` and `g/b` are more meaningful.
- The minimum green-dominance ratio of a blank green screen appears to behave like a screen-specific parameter.

So the core project conclusion is:

A practical chroma keyer should use green dominance, preferably in ratio form, and refine it with a notion of what shades are genuinely green.

In [5]:
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from ipywidgets import FloatSlider, IntSlider, Button, FileUpload, Output, VBox, HBox, HTML, Dropdown
from IPython.display import display, clear_output
import io

print("=== Project Report Implementation ===")
print("Compare the main report approaches on one uploaded image.\n")

file_upload_report = FileUpload(accept='image/*', multiple=False, description='Upload Image')
process_btn_report = Button(description='Run Report Methods', button_style='info')
output_display_report = Output()
current_image_report = {'data': None, 'name': None}

method_report = Dropdown(
    options=[
        ('Histogram Screening', 'histogram'),
        ('RGB Bounds (Grande)', 'rgb_bounds'),
        ('Ratio Bound (Venti)', 'ratio'),
        ('Intrinsic Ratio Bound (Trenta)', 'intrinsic_ratio'),
    ],
    value='ratio',
    description='Method:'
)

border_frac_report = FloatSlider(value=0.10, min=0.02, max=0.25, step=0.01, description='Border:')
g_min_report = FloatSlider(value=0.35, min=0.0, max=1.0, step=0.01, description='G min:')
r_max_report = FloatSlider(value=0.50, min=0.0, max=1.0, step=0.01, description='R max:')
b_max_report = FloatSlider(value=0.50, min=0.0, max=1.0, step=0.01, description='B max:')
super_bound_report = FloatSlider(value=1.20, min=1.0, max=2.0, step=0.05, description='Ratio:')
safety_factor_report = FloatSlider(value=0.90, min=0.70, max=1.00, step=0.01, description='Safety:')
hist_width_report = IntSlider(value=18, min=4, max=50, step=1, description='Hist width:')

def get_uploaded_file(upload_value):
    return list(upload_value.values())[0] if hasattr(upload_value, 'values') else upload_value[0]

def to_rgb(img):
    return np.array(img).astype(np.float32)[:, :, :3] / 255.0

def border_pixels(rgb, frac):
    h, w, _ = rgb.shape
    bh = max(1, int(h * frac))
    bw = max(1, int(w * frac))
    top = rgb[:bh, :, :].reshape(-1, 3)
    bottom = rgb[-bh:, :, :].reshape(-1, 3)
    left = rgb[bh:h-bh, :bw, :].reshape(-1, 3) if h > 2 * bh else np.empty((0, 3))
    right = rgb[bh:h-bh, -bw:, :].reshape(-1, 3) if h > 2 * bh else np.empty((0, 3))
    return np.vstack([top, bottom, left, right])

def apply_histogram_screening(rgb, frac=0.10, width=18):
    border = border_pixels(rgb, frac)
    mask_channels = []
    ranges = []
    for c in range(3):
        hist, edges = np.histogram(border[:, c], bins=256, range=(0, 1))
        peak = int(np.argmax(hist))
        lo = max(0, peak - width)
        hi = min(255, peak + width)
        low = edges[lo]
        high = edges[hi + 1]
        mask_channels.append((rgb[:, :, c] >= low) & (rgb[:, :, c] <= high))
        ranges.append((low, high))
    mask = mask_channels[0] & mask_channels[1] & mask_channels[2]
    return mask, ranges

def apply_rgb_bounds(rgb, g_min=0.35, r_max=0.50, b_max=0.50):
    r = rgb[:, :, 0]
    g = rgb[:, :, 1]
    b = rgb[:, :, 2]
    return (g > g_min) & (r < r_max) & (b < b_max) & (g > r) & (g > b)

def apply_ratio_bound(rgb, ratio=1.20, g_floor=0.14):
    eps = 1e-6
    r = rgb[:, :, 0]
    g = rgb[:, :, 1]
    b = rgb[:, :, 2]
    return (g > g_floor) & (g > r) & (g > b) & ((g / (r + eps)) > ratio) & ((g / (b + eps)) > ratio)

def estimate_intrinsic_bound(rgb, frac=0.10, safety=0.90):
    eps = 1e-6
    border = border_pixels(rgb, frac)
    g = border[:, 1]
    r = border[:, 0]
    b = border[:, 2]
    gr_min = np.min(g / (r + eps))
    gb_min = np.min(g / (b + eps))
    bound = safety * min(gr_min, gb_min)
    return bound, gr_min, gb_min

def apply_intrinsic_ratio_bound(rgb, frac=0.10, safety=0.90, g_floor=0.14):
    bound, gr_min, gb_min = estimate_intrinsic_bound(rgb, frac=frac, safety=safety)
    mask = apply_ratio_bound(rgb, ratio=bound, g_floor=g_floor)
    return mask, bound, gr_min, gb_min

def process_report_methods():
    if current_image_report['data'] is None:
        with output_display_report:
            clear_output()
            print('Please upload an image first')
        return

    rgb = to_rgb(current_image_report['data'])
    method = method_report.value
    details = []

    if method == 'histogram':
        mask, ranges = apply_histogram_screening(rgb, frac=border_frac_report.value, width=hist_width_report.value)
        details.append(f"Estimated histogram ranges: R[{ranges[0][0]:.3f}, {ranges[0][1]:.3f}], G[{ranges[1][0]:.3f}, {ranges[1][1]:.3f}], B[{ranges[2][0]:.3f}, {ranges[2][1]:.3f}]")
        method_title = 'Histogram Screening'
    elif method == 'rgb_bounds':
        mask = apply_rgb_bounds(rgb, g_min=g_min_report.value, r_max=r_max_report.value, b_max=b_max_report.value)
        details.append(f"Rule used: g > {g_min_report.value:.2f}, r < {r_max_report.value:.2f}, b < {b_max_report.value:.2f}")
        method_title = 'RGB Bounds (Grande)'
    elif method == 'ratio':
        mask = apply_ratio_bound(rgb, ratio=super_bound_report.value)
        details.append(f"Rule used: g/r > {super_bound_report.value:.2f} and g/b > {super_bound_report.value:.2f}")
        method_title = 'Ratio Bound (Venti)'
    else:
        mask, bound, gr_min, gb_min = apply_intrinsic_ratio_bound(rgb, frac=border_frac_report.value, safety=safety_factor_report.value)
        details.append(f"Estimated min(g/r) on border = {gr_min:.3f}")
        details.append(f"Estimated min(g/b) on border = {gb_min:.3f}")
        details.append(f"Intrinsic bound used = {bound:.3f} (safety factor {safety_factor_report.value:.2f})")
        method_title = 'Intrinsic Ratio Bound (Trenta)'

    result = rgb.copy()
    result[mask] = [1.0, 1.0, 1.0]

    with output_display_report:
        clear_output()
        fig, axes = plt.subplots(1, 3, figsize=(15, 5))
        axes[0].imshow(rgb)
        axes[0].set_title('Original')
        axes[0].axis('off')

        axes[1].imshow(mask, cmap='gray')
        axes[1].set_title('Background Mask')
        axes[1].axis('off')

        axes[2].imshow(result)
        axes[2].set_title(method_title)
        axes[2].axis('off')

        plt.tight_layout()
        plt.show()

        total = mask.size
        keyed = int(mask.sum())
        print(f"Method: {method_title}")
        print(f"Image: {current_image_report['name']}")
        print(f"Removed pixels: {keyed:,} / {total:,} ({100 * keyed / total:.2f}%)")
        print('\nMathematical summary:')
        if method == 'histogram':
            print('f(r,g,b)=1 when each channel lies in the estimated background interval from the border histograms.')
        elif method == 'rgb_bounds':
            print('f(r,g,b)=1 when g is large and r,b are small.')
        elif method == 'ratio':
            print('f(r,g,b)=1 when g/r and g/b both exceed the super-bound.')
        else:
            print('f(r,g,b)=1 when g/r and g/b exceed a bound estimated from the border green-screen itself.')
        print('\nDetails:')
        for line in details:
            print('-', line)

def handle_file_upload_report(change):
    if file_upload_report.value:
        uploaded_file = get_uploaded_file(file_upload_report.value)
        img = Image.open(io.BytesIO(uploaded_file['content'])).convert('RGBA')
        current_image_report['data'] = img
        current_image_report['name'] = uploaded_file['name']
        with output_display_report:
            clear_output()
            print(f"Image loaded: {uploaded_file['name']} ({img.size[0]}x{img.size[1]})")
            print('Choose a method and click Run Report Methods.')

def handle_process_click_report(button):
    process_report_methods()

file_upload_report.observe(handle_file_upload_report, names='value')
process_btn_report.on_click(handle_process_click_report)

controls_report = VBox([
    HTML('<b>Project Report Implementation</b>'),
    HTML('This cell implements the main mathematical ideas from the report and lets you compare them on an image.'),
    method_report,
    border_frac_report,
    hist_width_report,
    g_min_report,
    r_max_report,
    b_max_report,
    super_bound_report,
    safety_factor_report,
    file_upload_report,
    process_btn_report,
])

display(HBox([controls_report, output_display_report]))

print('Instructions:')
print('1. Upload an image from the project.')
print('2. Select one report approach.')
print('3. Tune its parameters if needed.')
print('4. Compare the mask and output image with the mathematical rule printed below the plots.')

=== Project Report Implementation ===
Compare the main report approaches on one uploaded image.



Instructions:
1. Upload an image from the project.
2. Select one report approach.
3. Tune its parameters if needed.
4. Compare the mask and output image with the mathematical rule printed below the plots.


## Boundary-Separation Approach

The earlier threshold rules are not explicitly learning a separating line from the RGB scatter plot. They mostly use axis-aligned bounds or ratio conditions.

This approach tries something closer to what the scatter plot suggests: use Approach 3 to get seed background pixels, get a rough foreground seed, then learn a **linear separator** in the `(R,G)` plane using a Fisher-style linear discriminant.

In [9]:
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from ipywidgets import FloatSlider, Button, FileUpload, Output, VBox, HBox, HTML
from IPython.display import display, clear_output
import io

print("=== Boundary-Separation Chroma Keying ===")

file_upload_boundary = FileUpload(accept='image/*', multiple=False, description='Upload Image')
process_btn_boundary = Button(description='Run Boundary Method', button_style='info')
output_display_boundary = Output()
current_image_boundary = {'data': None, 'name': None}

green_lower_boundary = FloatSlider(value=0.28, min=0, max=1, step=0.01, description='Green Min:')
green_upper_boundary = FloatSlider(value=0.97, min=0, max=1, step=0.01, description='Green Max:')
grey_bound_boundary = FloatSlider(value=0.00, min=0, max=1, step=0.005, description='Diff Min:')
score_shift_boundary = FloatSlider(value=0.00, min=-0.50, max=0.50, step=0.01, description='Bias Shift:')

def get_uploaded_file(upload_value):
    return list(upload_value.values())[0] if hasattr(upload_value, 'values') else upload_value[0]

def to_rgb(img):
    return np.array(img).astype(np.float32)[:, :, :3] / 255.0

def approach3_mask(rgb, green_lower, green_upper, grey_bound):
    r = rgb[:, :, 0]
    g = rgb[:, :, 1]
    b = rgb[:, :, 2]
    return (
        (np.argmax(rgb, axis=2) == 1)
        & (g > green_lower)
        & (g <= green_upper)
        & (np.abs(r - b) >= grey_bound)
        & (np.abs(r - g) >= grey_bound)
        & (np.abs(b - g) >= grey_bound)
    )

def fit_linear_separator_rg(bg_points, fg_points):
    mu_bg = bg_points.mean(axis=0)
    mu_fg = fg_points.mean(axis=0)
    cov_bg = np.cov((bg_points - mu_bg).T)
    cov_fg = np.cov((fg_points - mu_fg).T)
    sw = cov_bg + cov_fg + 1e-6 * np.eye(2)
    w = np.linalg.solve(sw, mu_bg - mu_fg)
    c = -0.5 * w @ (mu_bg + mu_fg)
    return w, c, mu_bg, mu_fg

def process_boundary_method():
    if current_image_boundary['data'] is None:
        with output_display_boundary:
            clear_output()
            print('Please upload an image first')
        return

    rgb = to_rgb(current_image_boundary['data'])
    seed_bg = approach3_mask(rgb, green_lower_boundary.value, green_upper_boundary.value, grey_bound_boundary.value)
    r = rgb[:, :, 0]
    g = rgb[:, :, 1]
    fg_seed = (~seed_bg) & ((g <= np.maximum(r, rgb[:, :, 2])) | (g < green_lower_boundary.value))

    bg_points = np.column_stack([r[seed_bg], g[seed_bg]])
    fg_points = np.column_stack([r[fg_seed], g[fg_seed]])

    if bg_points.shape[0] < 20 or fg_points.shape[0] < 20:
        with output_display_boundary:
            clear_output()
            print('Not enough seed pixels to learn a separator. Adjust the seed thresholds.')
        return

    sample_bg = bg_points[np.random.choice(bg_points.shape[0], size=min(6000, bg_points.shape[0]), replace=False)]
    sample_fg = fg_points[np.random.choice(fg_points.shape[0], size=min(6000, fg_points.shape[0]), replace=False)]
    w, c, mu_bg, mu_fg = fit_linear_separator_rg(sample_bg, sample_fg)

    rg = np.column_stack([r.reshape(-1), g.reshape(-1)])
    scores = (rg @ w + c + score_shift_boundary.value).reshape(r.shape)
    final_mask = (scores > 0) & (g > r) & (g > rgb[:, :, 2])

    result = rgb.copy()
    result[final_mask] = [1.0, 1.0, 1.0]

    with output_display_boundary:
        clear_output()
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))
        axes[0, 0].imshow(rgb)
        axes[0, 0].set_title('Original Image')
        axes[0, 0].axis('off')

        axes[0, 1].imshow(seed_bg, cmap='gray')
        axes[0, 1].set_title('Approach 3 Seed Background')
        axes[0, 1].axis('off')

        axes[1, 0].imshow(final_mask, cmap='gray')
        axes[1, 0].set_title('Boundary-Based Final Mask')
        axes[1, 0].axis('off')

        axes[1, 1].imshow(result)
        axes[1, 1].set_title('Chroma Key Result')
        axes[1, 1].axis('off')
        plt.tight_layout()
        plt.show()

        fig2, ax = plt.subplots(figsize=(7, 6))
        ax.scatter(sample_fg[:, 0], sample_fg[:, 1], s=4, alpha=0.3, label='Foreground seed', c='tab:brown')
        ax.scatter(sample_bg[:, 0], sample_bg[:, 1], s=4, alpha=0.3, label='Background seed', c='tab:green')
        xs = np.linspace(0, 1, 200)
        if abs(w[1]) > 1e-8:
            ys = -(w[0] * xs + c + score_shift_boundary.value) / w[1]
            ax.plot(xs, ys, color='black', linewidth=2, label='Learned separator')
        ax.set_xlabel('Red')
        ax.set_ylabel('Green')
        ax.set_title('RG Scatter with Learned Boundary')
        ax.legend()
        ax.grid(alpha=0.3)
        plt.show()

        print('Interpretation: this method explicitly learns a line in the (R,G) plane.')
        print(f'Background pixels removed: {int(final_mask.sum()):,} / {final_mask.size:,} ({100 * final_mask.mean():.2f}%)')
        print(f'Learned score: w_R={w[0]:.3f}, w_G={w[1]:.3f}, intercept={c + score_shift_boundary.value:.3f}')

def handle_file_upload_boundary(change):
    if file_upload_boundary.value:
        uploaded_file = get_uploaded_file(file_upload_boundary.value)
        img = Image.open(io.BytesIO(uploaded_file['content'])).convert('RGBA')
        current_image_boundary['data'] = img
        current_image_boundary['name'] = uploaded_file['name']
        with output_display_boundary:
            clear_output()
            print(f"Image loaded: {uploaded_file['name']} ({img.size[0]}x{img.size[1]})")
            print('Run the boundary method to learn a separator from seed pixels.')

file_upload_boundary.observe(handle_file_upload_boundary, names='value')
process_btn_boundary.on_click(lambda _: process_boundary_method())

controls_boundary = VBox([
    HTML('<b>Boundary-Separation Method</b>'),
    HTML('Use Approach 3 as a seed generator, then learn a linear separator in the RG plane.'),
    green_lower_boundary,
    green_upper_boundary,
    grey_bound_boundary,
    score_shift_boundary,
    file_upload_boundary,
    process_btn_boundary,
])

display(HBox([controls_boundary, output_display_boundary]))


=== Boundary-Separation Chroma Keying ===


## Neighborhood Greenness Approach

This approach classifies a pixel using the **average color of its local `k x k` neighborhood** rather than only the pixel itself. That makes the decision less noisy and uses local context.

A second variant applies PCA to these local-average colors, so PCA is used on neighborhood summaries instead of raw pixels.

In [ ]:
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from ipywidgets import IntSlider, FloatSlider, Dropdown, Button, FileUpload, Output, VBox, HBox, HTML
from IPython.display import display, clear_output
import io

print("=== Neighborhood Greenness Chroma Keying ===")

file_upload_local = FileUpload(accept='image/*', multiple=False, description='Upload Image')
process_btn_local = Button(description='Run Neighborhood Method', button_style='info')
output_display_local = Output()
current_image_local = {'data': None, 'name': None}

local_radius = IntSlider(value=3, min=1, max=8, step=1, description='Radius:')
local_mode = Dropdown(options=[('Local thresholding', 'threshold'), ('Local-average PCA', 'pca')], value='threshold', description='Mode:')
green_lower_local = FloatSlider(value=0.28, min=0, max=1, step=0.01, description='Green Min:')
green_upper_local = FloatSlider(value=0.97, min=0, max=1, step=0.01, description='Green Max:')
grey_bound_local = FloatSlider(value=0.00, min=0, max=1, step=0.005, description='Diff Min:')
distance_threshold_local = FloatSlider(value=2.50, min=0.50, max=6.00, step=0.10, description='PCA Thresh:')

def get_uploaded_file(upload_value):
    return list(upload_value.values())[0] if hasattr(upload_value, 'values') else upload_value[0]

def to_rgb(img):
    return np.array(img).astype(np.float32)[:, :, :3] / 255.0

def box_mean_2d(channel, radius):
    padded = np.pad(channel, ((radius, radius), (radius, radius)), mode='edge')
    integral = np.pad(padded, ((1, 0), (1, 0)), mode='constant').cumsum(axis=0).cumsum(axis=1)
    size = 2 * radius + 1
    total = integral[size:, size:] - integral[:-size, size:] - integral[size:, :-size] + integral[:-size, :-size]
    return total / (size * size)

def local_mean_rgb(rgb, radius):
    return np.stack([box_mean_2d(rgb[:, :, i], radius) for i in range(3)], axis=2)

def local_threshold_mask(local_rgb, green_lower, green_upper, grey_bound):
    r = local_rgb[:, :, 0]
    g = local_rgb[:, :, 1]
    b = local_rgb[:, :, 2]
    return (
        (np.argmax(local_rgb, axis=2) == 1)
        & (g > green_lower)
        & (g <= green_upper)
        & (np.abs(r - b) >= grey_bound)
        & (np.abs(r - g) >= grey_bound)
        & (np.abs(b - g) >= grey_bound)
    )

def fit_pca(points):
    mean = points.mean(axis=0)
    centered = points - mean
    cov = np.cov(centered, rowvar=False)
    eigvals, eigvecs = np.linalg.eigh(cov)
    order = np.argsort(eigvals)[::-1]
    eigvals = eigvals[order]
    eigvecs = eigvecs[:, order]
    stds = np.sqrt(np.maximum(eigvals, 1e-8))
    return mean, eigvecs, stds

def pca_distance(points, mean, eigvecs, stds):
    centered = points - mean
    proj = centered @ eigvecs
    proj = proj / stds
    return np.sqrt(np.sum(proj ** 2, axis=1))

def process_local_method():
    if current_image_local['data'] is None:
        with output_display_local:
            clear_output()
            print('Please upload an image first')
        return

    rgb = to_rgb(current_image_local['data'])
    local_rgb = local_mean_rgb(rgb, local_radius.value)
    seed_mask = local_threshold_mask(local_rgb, green_lower_local.value, green_upper_local.value, grey_bound_local.value)

    if local_mode.value == 'threshold':
        final_mask = seed_mask
        method_label = 'Local thresholding'
    else:
        fit_pixels = local_rgb[seed_mask]
        if fit_pixels.shape[0] < 25:
            with output_display_local:
                clear_output()
                print('Too few local-average background pixels for PCA. Relax the local thresholds.')
            return
        mean, eigvecs, stds = fit_pca(fit_pixels)
        distances = pca_distance(local_rgb.reshape(-1, 3), mean, eigvecs, stds).reshape(local_rgb.shape[:2])
        final_mask = distances <= distance_threshold_local.value
        method_label = 'Local-average PCA'

    result = rgb.copy()
    result[final_mask] = [1.0, 1.0, 1.0]

    with output_display_local:
        clear_output()
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))
        axes[0, 0].imshow(rgb)
        axes[0, 0].set_title('Original Image')
        axes[0, 0].axis('off')

        axes[0, 1].imshow(local_rgb[:, :, 1], cmap='Greens')
        axes[0, 1].set_title('Local Mean Green Channel')
        axes[0, 1].axis('off')

        axes[1, 0].imshow(final_mask, cmap='gray')
        axes[1, 0].set_title(f'{method_label} Mask')
        axes[1, 0].axis('off')

        axes[1, 1].imshow(result)
        axes[1, 1].set_title('Chroma Key Result')
        axes[1, 1].axis('off')
        plt.tight_layout()
        plt.show()

        print(f'Method: {method_label}')
        print(f'Neighborhood size: {(2 * local_radius.value + 1)} x {(2 * local_radius.value + 1)}')
        print(f'Pixels removed: {int(final_mask.sum()):,} / {final_mask.size:,} ({100 * final_mask.mean():.2f}%)')
        if local_mode.value == 'pca':
            print(f'Seed pixels used to fit PCA on local averages: {fit_pixels.shape[0]:,}')

def handle_file_upload_local(change):
    if file_upload_local.value:
        uploaded_file = get_uploaded_file(file_upload_local.value)
        img = Image.open(io.BytesIO(uploaded_file['content'])).convert('RGBA')
        current_image_local['data'] = img
        current_image_local['name'] = uploaded_file['name']
        with output_display_local:
            clear_output()
            print(f"Image loaded: {uploaded_file['name']} ({img.size[0]}x{img.size[1]})")

file_upload_local.observe(handle_file_upload_local, names='value')
process_btn_local.on_click(lambda _: process_local_method())

controls_local = VBox([
    HTML('<b>Neighborhood Greenness Method</b>'),
    HTML('Classify pixels using local color averages, with an optional PCA model fit on those local averages.'),
    local_radius,
    local_mode,
    green_lower_local,
    green_upper_local,
    grey_bound_local,
    distance_threshold_local,
    file_upload_local,
    process_btn_local,
])

display(HBox([controls_local, output_display_local]))


=== Neighborhood Greenness Chroma Keying ===


## Edge-Aware Computer Vision Approach

HOG itself is more of a feature descriptor than a direct background-removal rule, so here we use a closely related idea: **gradient/edge information** to protect strong object boundaries while removing green background pixels.

This gives an edge-aware chroma keyer: start with a green mask, find strong edges, then avoid deleting pixels near those edges.

In [ ]:
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from ipywidgets import IntSlider, FloatSlider, Button, FileUpload, Output, VBox, HBox, HTML
from IPython.display import display, clear_output
import io

print("=== Edge-Aware Computer Vision Chroma Keying ===")

file_upload_edge = FileUpload(accept='image/*', multiple=False, description='Upload Image')
process_btn_edge = Button(description='Run Edge-Aware Method', button_style='info')
output_display_edge = Output()
current_image_edge = {'data': None, 'name': None}

green_lower_edge = FloatSlider(value=0.28, min=0, max=1, step=0.01, description='Green Min:')
green_upper_edge = FloatSlider(value=0.97, min=0, max=1, step=0.01, description='Green Max:')
grey_bound_edge = FloatSlider(value=0.00, min=0, max=1, step=0.005, description='Diff Min:')
edge_threshold = FloatSlider(value=0.10, min=0.01, max=0.50, step=0.01, description='Edge Thresh:')
protect_radius = IntSlider(value=2, min=0, max=8, step=1, description='Protect Rad:')

def get_uploaded_file(upload_value):
    return list(upload_value.values())[0] if hasattr(upload_value, 'values') else upload_value[0]

def to_rgb(img):
    return np.array(img).astype(np.float32)[:, :, :3] / 255.0

def approach3_mask(rgb, green_lower, green_upper, grey_bound):
    r = rgb[:, :, 0]
    g = rgb[:, :, 1]
    b = rgb[:, :, 2]
    return (
        (np.argmax(rgb, axis=2) == 1)
        & (g > green_lower)
        & (g <= green_upper)
        & (np.abs(r - b) >= grey_bound)
        & (np.abs(r - g) >= grey_bound)
        & (np.abs(b - g) >= grey_bound)
    )

def box_mean_2d(channel, radius):
    padded = np.pad(channel, ((radius, radius), (radius, radius)), mode='edge')
    integral = np.pad(padded, ((1, 0), (1, 0)), mode='constant').cumsum(axis=0).cumsum(axis=1)
    size = 2 * radius + 1
    total = integral[size:, size:] - integral[:-size, size:] - integral[size:, :-size] + integral[:-size, :-size]
    return total / (size * size)

def sobel_edges(gray):
    p = np.pad(gray, 1, mode='edge')
    gx = (
        -p[:-2, :-2] - 2 * p[1:-1, :-2] - p[2:, :-2]
        + p[:-2, 2:] + 2 * p[1:-1, 2:] + p[2:, 2:]
    )
    gy = (
        -p[:-2, :-2] - 2 * p[:-2, 1:-1] - p[:-2, 2:]
        + p[2:, :-2] + 2 * p[2:, 1:-1] + p[2:, 2:]
    )
    return np.sqrt(gx ** 2 + gy ** 2)

def process_edge_method():
    if current_image_edge['data'] is None:
        with output_display_edge:
            clear_output()
            print('Please upload an image first')
        return

    rgb = to_rgb(current_image_edge['data'])
    base_mask = approach3_mask(rgb, green_lower_edge.value, green_upper_edge.value, grey_bound_edge.value)
    gray = 0.299 * rgb[:, :, 0] + 0.587 * rgb[:, :, 1] + 0.114 * rgb[:, :, 2]
    edge_mag = sobel_edges(gray)
    edge_seed = edge_mag > edge_threshold.value
    edge_protect = box_mean_2d(edge_seed.astype(np.float32), protect_radius.value) > 0 if protect_radius.value > 0 else edge_seed
    final_mask = base_mask & (~edge_protect)

    result = rgb.copy()
    result[final_mask] = [1.0, 1.0, 1.0]

    with output_display_edge:
        clear_output()
        fig, axes = plt.subplots(2, 3, figsize=(16, 10))
        axes[0, 0].imshow(rgb)
        axes[0, 0].set_title('Original Image')
        axes[0, 0].axis('off')

        axes[0, 1].imshow(base_mask, cmap='gray')
        axes[0, 1].set_title('Base Green Mask')
        axes[0, 1].axis('off')

        axes[0, 2].imshow(edge_mag, cmap='magma')
        axes[0, 2].set_title('Gradient Magnitude')
        axes[0, 2].axis('off')

        axes[1, 0].imshow(edge_protect, cmap='gray')
        axes[1, 0].set_title('Protected Edge Region')
        axes[1, 0].axis('off')

        axes[1, 1].imshow(final_mask, cmap='gray')
        axes[1, 1].set_title('Edge-Aware Final Mask')
        axes[1, 1].axis('off')

        axes[1, 2].imshow(result)
        axes[1, 2].set_title('Chroma Key Result')
        axes[1, 2].axis('off')
        plt.tight_layout()
        plt.show()

        print('This is an edge-aware CV approach: remove green pixels, but keep pixels near strong boundaries.')
        print(f'Base mask removed: {int(base_mask.sum()):,} / {base_mask.size:,} ({100 * base_mask.mean():.2f}%)')
        print(f'Final mask removed: {int(final_mask.sum()):,} / {final_mask.size:,} ({100 * final_mask.mean():.2f}%)')

def handle_file_upload_edge(change):
    if file_upload_edge.value:
        uploaded_file = get_uploaded_file(file_upload_edge.value)
        img = Image.open(io.BytesIO(uploaded_file['content'])).convert('RGBA')
        current_image_edge['data'] = img
        current_image_edge['name'] = uploaded_file['name']
        with output_display_edge:
            clear_output()
            print(f"Image loaded: {uploaded_file['name']} ({img.size[0]}x{img.size[1]})")

file_upload_edge.observe(handle_file_upload_edge, names='value')
process_btn_edge.on_click(lambda _: process_edge_method())

controls_edge = VBox([
    HTML('<b>Edge-Aware CV Method</b>'),
    HTML('Use strong image gradients to protect boundaries while removing green-screen pixels.'),
    green_lower_edge,
    green_upper_edge,
    grey_bound_edge,
    edge_threshold,
    protect_radius,
    file_upload_edge,
    process_btn_edge,
])

display(HBox([controls_edge, output_display_edge]))


=== Edge-Aware Computer Vision Chroma Keying ===


## Manual Refinement Tools

After automatic chroma keying, a user may still want to refine the result manually. This section adds practical post-processing tools:

- **Boundary grow/shrink**: increase or decrease the background mask pixel-wise near object boundaries
- **Green-screen eraser**: paint more pixels into the background mask
- **Image restore**: paint pixels back into the foreground

If `M` is the background mask, then increasing the boundary corresponds to a dilation of `M`, and decreasing the boundary corresponds to an erosion of `M`. These are standard segmentation-refinement tools.

In [13]:
try:
    get_ipython().run_line_magic('matplotlib', 'widget')
except Exception:
    pass

import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from ipywidgets import FloatSlider, IntSlider, Dropdown, Button, FileUpload, Output, VBox, HBox, HTML
from IPython.display import display, clear_output
import io

print("=== Manual Refinement Tools ===")
print("Pixel-wise boundary refinement and manual mask painting.\n")

file_upload_refine = FileUpload(accept='image/*', multiple=False, description='Upload Image')
init_btn_refine = Button(description='Initialize Mask', button_style='success')
reset_btn_refine = Button(description='Reset Edits')
refresh_btn_refine = Button(description='Refresh View', button_style='info')
output_display_refine = Output()

green_lower_refine = FloatSlider(value=0.28, min=0, max=1, step=0.01, description='Green Min:')
green_upper_refine = FloatSlider(value=0.97, min=0, max=1, step=0.01, description='Green Max:')
grey_bound_refine = FloatSlider(value=0.00, min=0, max=1, step=0.005, description='Diff Min:')
boundary_adjust_refine = IntSlider(value=0, min=-8, max=8, step=1, description='Boundary:')
brush_radius_refine = IntSlider(value=12, min=1, max=60, step=1, description='Brush:')
tool_mode_refine = Dropdown(
    options=[('Erase Green Screen', 'erase_green'), ('Restore Image', 'restore_image')],
    value='erase_green',
    description='Tool:'
)
output_mode_refine = Dropdown(
    options=[('White background', 'white'), ('Transparent background', 'transparent')],
    value='white',
    description='Output:'
)

refine_state = {
    'image': None,
    'name': None,
    'rgb': None,
    'base_mask': None,
    'edited_mask': None,
    'figure': None,
    'axes': None,
    'cid': None,
}

def get_uploaded_file(upload_value):
    return list(upload_value.values())[0] if hasattr(upload_value, 'values') else upload_value[0]

def to_rgb(img):
    return np.array(img).astype(np.float32)[:, :, :3] / 255.0

def approach3_mask(rgb, green_lower, green_upper, grey_bound):
    r = rgb[:, :, 0]
    g = rgb[:, :, 1]
    b = rgb[:, :, 2]
    return (
        (np.argmax(rgb, axis=2) == 1)
        & (g > green_lower)
        & (g <= green_upper)
        & (np.abs(r - b) >= grey_bound)
        & (np.abs(r - g) >= grey_bound)
        & (np.abs(b - g) >= grey_bound)
    )

def box_mean_2d(channel, radius):
    if radius <= 0:
        return channel.astype(np.float32)
    padded = np.pad(channel, ((radius, radius), (radius, radius)), mode='edge')
    integral = np.pad(padded, ((1, 0), (1, 0)), mode='constant').cumsum(axis=0).cumsum(axis=1)
    size = 2 * radius + 1
    total = integral[size:, size:] - integral[:-size, size:] - integral[size:, :-size] + integral[:-size, :-size]
    return total / (size * size)

def binary_dilate(mask, radius):
    if radius <= 0:
        return mask.copy()
    return box_mean_2d(mask.astype(np.float32), radius) > 0

def binary_erode(mask, radius):
    if radius <= 0:
        return mask.copy()
    return box_mean_2d(mask.astype(np.float32), radius) >= (1.0 - 1e-6)

def apply_boundary_adjust(mask, adjust):
    if adjust > 0:
        return binary_dilate(mask, adjust)
    if adjust < 0:
        return binary_erode(mask, -adjust)
    return mask.copy()

def build_result(rgb, mask, output_mode):
    if output_mode == 'transparent':
        alpha = np.ones((rgb.shape[0], rgb.shape[1], 1), dtype=np.float32)
        alpha[mask] = 0.0
        return np.dstack([rgb, alpha])
    result = rgb.copy()
    result[mask] = [1.0, 1.0, 1.0]
    return result

def current_display_mask():
    if refine_state['edited_mask'] is None:
        return None
    return apply_boundary_adjust(refine_state['edited_mask'], boundary_adjust_refine.value)

def on_click_refine(event):
    if refine_state['edited_mask'] is None or refine_state['axes'] is None:
        return
    if event.inaxes != refine_state['axes'][1]:
        return
    if event.xdata is None or event.ydata is None:
        return

    y = int(round(event.ydata))
    x = int(round(event.xdata))
    h, w = refine_state['edited_mask'].shape
    yy, xx = np.ogrid[:h, :w]
    rr = brush_radius_refine.value
    brush = (yy - y) ** 2 + (xx - x) ** 2 <= rr ** 2

    if tool_mode_refine.value == 'erase_green':
        refine_state['edited_mask'][brush] = True
    else:
        refine_state['edited_mask'][brush] = False

    show_refine_result()

def show_refine_result():
    if refine_state['rgb'] is None or refine_state['edited_mask'] is None:
        return

    rgb = refine_state['rgb']
    display_mask = current_display_mask()
    result = build_result(rgb, display_mask, output_mode_refine.value)
    overlay = rgb.copy()
    overlay[display_mask] = 0.65 * overlay[display_mask] + 0.35 * np.array([0.0, 1.0, 0.0])

    with output_display_refine:
        clear_output(wait=True)
        fig, axes = plt.subplots(1, 3, figsize=(16, 5))
        axes[0].imshow(rgb)
        axes[0].set_title('Original Image')
        axes[0].axis('off')

        axes[1].imshow(overlay)
        axes[1].set_title('Editable Mask Overlay')
        axes[1].axis('off')

        axes[2].imshow(result)
        axes[2].set_title('Refined Result')
        axes[2].axis('off')
        plt.tight_layout()
        plt.show()

        refine_state['figure'] = fig
        refine_state['axes'] = axes
        if refine_state['cid'] is not None:
            try:
                fig.canvas.mpl_disconnect(refine_state['cid'])
            except Exception:
                pass
        refine_state['cid'] = fig.canvas.mpl_connect('button_press_event', on_click_refine)

        print(f"Current image: {refine_state['name']}")
        print(f"Removed pixels: {int(display_mask.sum()):,} / {display_mask.size:,} ({100 * display_mask.mean():.2f}%)")
        print('Boundary meaning: positive values remove more around edges; negative values restore more foreground edges.')
        print('Click on the middle panel to paint edits.')

def initialize_mask_refine(button=None):
    if refine_state['image'] is None:
        with output_display_refine:
            clear_output()
            print('Please upload an image first.')
        return

    rgb = to_rgb(refine_state['image'])
    base_mask = approach3_mask(rgb, green_lower_refine.value, green_upper_refine.value, grey_bound_refine.value)
    refine_state['rgb'] = rgb
    refine_state['base_mask'] = base_mask.copy()
    refine_state['edited_mask'] = base_mask.copy()
    show_refine_result()

def reset_refine_result(button=None):
    if refine_state['base_mask'] is not None:
        refine_state['edited_mask'] = refine_state['base_mask'].copy()
        show_refine_result()

def handle_file_upload_refine(change):
    if file_upload_refine.value:
        uploaded_file = get_uploaded_file(file_upload_refine.value)
        img = Image.open(io.BytesIO(uploaded_file['content'])).convert('RGBA')
        refine_state['image'] = img
        refine_state['name'] = uploaded_file['name']
        refine_state['rgb'] = None
        refine_state['base_mask'] = None
        refine_state['edited_mask'] = None
        with output_display_refine:
            clear_output()
            print(f"Image loaded: {uploaded_file['name']} ({img.size[0]}x{img.size[1]})")
            print('Click Initialize Mask to start refinement.')

file_upload_refine.observe(handle_file_upload_refine, names='value')
init_btn_refine.on_click(initialize_mask_refine)
reset_btn_refine.on_click(reset_refine_result)
refresh_btn_refine.on_click(lambda _: show_refine_result())
boundary_adjust_refine.observe(lambda change: show_refine_result() if refine_state['edited_mask'] is not None else None, names='value')
output_mode_refine.observe(lambda change: show_refine_result() if refine_state['edited_mask'] is not None else None, names='value')

controls_refine = VBox([
    HTML('<b>Manual Refinement Tools</b>'),
    HTML('Initialize a mask, then use boundary grow/shrink and brush-based edits for manual cleanup.'),
    green_lower_refine,
    green_upper_refine,
    grey_bound_refine,
    tool_mode_refine,
    brush_radius_refine,
    boundary_adjust_refine,
    output_mode_refine,
    file_upload_refine,
    init_btn_refine,
    reset_btn_refine,
    refresh_btn_refine,
])

display(HBox([controls_refine, output_display_refine]))

print('Instructions:')
print('1. Upload an image and click Initialize Mask.')
print('2. Use Boundary to grow or shrink the background mask near object edges.')
print('3. Click on the middle panel to paint edits.')
print('4. Use Erase Green Screen to add pixels to the background mask, or Restore Image to bring them back.')
print('5. If click editing does not respond in your notebook frontend, rerun this cell after enabling a widget-capable matplotlib backend.')


=== Manual Refinement Tools ===
Pixel-wise boundary refinement and manual mask painting.



Instructions:
1. Upload an image and click Initialize Mask.
2. Use Boundary to grow or shrink the background mask near object edges.
3. Click on the middle panel to paint edits.
4. Use Erase Green Screen to add pixels to the background mask, or Restore Image to bring them back.
5. If click editing does not respond in your notebook frontend, rerun this cell after enabling a widget-capable matplotlib backend.
